In [1]:
import os
import json

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

In [2]:
# base_path = "../../../db/object-detection"
# base_path = "/mnt/pg-fs01/Current/DB/ObjectDetectionV3"
base_path = "../../../../data/ObjectDetectionV3_1"
df = pd.read_csv(os.path.join(base_path, "annotation.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))
df_imgs.head()

,id,name,width,height,batch,folder,task_id,job_id,job_url,group_id
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam2_H255_discaunt_29012024_2024-02-06-11h11m0...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,291e1aa6-1882-4eed-a749-ac5860383fa0
1,b74d0fb1-32e7-4629-8fad-c1a606cb0fb3,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,291e1aa6-1882-4eed-a749-ac5860383fa0
2,1a2a73ed-562b-4f79-8374-59eef50bea63,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,291e1aa6-1882-4eed-a749-ac5860383fa0
3,e2acf72f-9e57-4f7a-a0ee-89aed453dd32,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,291e1aa6-1882-4eed-a749-ac5860383fa0
4,29a3b2e9-5d65-4441-9588-42dea2bc372f,Cam2_H255_discaunt_29012024_2024-02-06-11h11m2...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,291e1aa6-1882-4eed-a749-ac5860383fa0


In [3]:
print("before", df_imgs.shape, df.shape)

# TODO: do not remove image without annotations when problem with batch4_22.5/5 will be fixed
# rm images without annotations
# tmp_df = pd.merge(df, df_imgs, how="outer", left_on="img_id", right_on="id")
# tmp_ids = tmp_df[tmp_df["id_x"].isna()]["id_y"].values
# df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
# print("number of images without annotations:", len(tmp_ids))

# rm images with problems in annotations
tmp_ids = df[(df["xtl"].isna()) | (df["x"].isna())]["img_id"].unique()
tmp_ids = df[(df["xtl"].isna())]["img_id"].unique()
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
df = df[~df["img_id"].isin(tmp_ids)]
print("number of images with 'problematic' annotations:", len(tmp_ids))

#rm duplicates
df1 = df_imgs.groupby(['name','batch', "folder", "id"]).size().reset_index()[['name','batch', "folder", "id"]]
duplicates = df1[df1.duplicated('name', keep=False)]
tmp_ids = duplicates.iloc[1::2]["id"].values
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
df = df[~df["img_id"].isin(tmp_ids)]
print("number of duplicates:", len(tmp_ids))

print("after", df_imgs.shape, df.shape)

before (710, 10) (21650, 16)
number of images with 'problematic' annotations: 39
number of duplicates: 0
after (671, 10) (20138, 16)


In [4]:
image_ids = df_imgs["id"].values
group_ids = df_imgs["group_id"].values
labels = sorted(df["label"].unique())

val_size: float = 0.1
randon_seed: int = 43
num_skip_valid_splits: int = 0

n_splits = round(1 / val_size)
gskf = GroupShuffleSplit(n_splits=n_splits, random_state=randon_seed)
current_try = 0
# Add check for classes and groups
for index, indices in enumerate(gskf.split(image_ids, groups=group_ids)):
    print(f"Trying split # {index}")
    train_indexes, test_indexes = indices
    train_ids = image_ids[train_indexes]
    test_ids = image_ids[test_indexes]
    train_classes = sorted(df[df["img_id"].isin(train_ids)]["label"].unique())
    test_classes = sorted(df[df["img_id"].isin(test_ids)]["label"].unique())

    is_satisfactory_split = labels == train_classes and labels == test_classes
    if is_satisfactory_split:
        if current_try >= num_skip_valid_splits:
            print("OK")
            break
        else:
            current_try += 1
    elif index == n_splits-1:
        print("not OK")
        break
    
print(f"train: {len(train_ids)} test: {len(test_ids)}")

with open(os.path.join(base_path, "train_test_split.json"), "w") as f:
    json.dump({"train": train_ids.tolist(), "test": test_ids.tolist()}, f)

Trying split # 0
Trying split # 1
Trying split # 2
Trying split # 3
Trying split # 4
Trying split # 5
Trying split # 6
Trying split # 7
Trying split # 8
Trying split # 9
not OK
train: 593 test: 78
